In [1]:
try:
    import transformers
    import datasets
    from transformers import BitsAndBytesConfig
    from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
    import evaluate
except:
    print("all or at least one of the package not installed. Installing now. Please remember to restart kernel once done")
    !pip install transformers bitsandbytes accelerate datasets peft evaluate 

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM
from datasets import Dataset as hf_Dataset

In [3]:
import torch
from torch.utils.data import DataLoader, Dataset

from functools import partial

# TODO: Import any packages that you might need
#importing the "King" library :P
import sagemaker
import boto3
from sagemaker.inputs import TrainingInput

#sagemaker pytorch container estimator related libraries
from sagemaker.pytorch import PyTorch
from sagemaker.pytorch import PyTorchModel


from sagemaker.estimator import Estimator
from sagemaker.model import Model
from sagemaker.predictor import Predictor

#general utility imports
import os
import glob
import pandas as pd
import numpy as np
import random

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/pydantic/_internal/_fields.py:192: UserWarning: Field name "json" in "MonitoringDatasetFormat" shadows an attribute in parent "Base"
  warnings.warn(


[03/03/25 10:23:03] INFO     Found credentials from IAM Role:                                   ]8;id=173809;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=764759;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [4]:
huggingface_dataset_name = "knkarthick/dialogsum"
ds = load_dataset(huggingface_dataset_name)
ds

README.md:   0%|          | 0.00/4.65k [00:00<?, ?B/s]

train.csv:   0%|          | 0.00/11.3M [00:00<?, ?B/s]

validation.csv:   0%|          | 0.00/442k [00:00<?, ?B/s]

test.csv:   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12460 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [5]:
ckpt = 'facebook/opt-350m'
orig_model = AutoModelForCausalLM.from_pretrained(ckpt)
tokenizer = AutoTokenizer.from_pretrained(ckpt)

config.json:   0%|          | 0.00/644 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/663M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/662M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

In [6]:
orig_model

OPTForCausalLM(
  (model): OPTModel(
    (decoder): OPTDecoder(
      (embed_tokens): Embedding(50272, 512, padding_idx=1)
      (embed_positions): OPTLearnedPositionalEmbedding(2050, 1024)
      (project_out): Linear(in_features=1024, out_features=512, bias=False)
      (project_in): Linear(in_features=512, out_features=1024, bias=False)
      (layers): ModuleList(
        (0-23): 24 x OPTDecoderLayer(
          (self_attn): OPTSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (activation_fn): ReLU()
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features

In [48]:
orig_model.config

OPTConfig {
  "_attn_implementation_autoset": true,
  "_name_or_path": "facebook/opt-350m",
  "_remove_final_layer_norm": false,
  "activation_dropout": 0.0,
  "activation_function": "relu",
  "architectures": [
    "OPTForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 2,
  "do_layer_norm_before": false,
  "dropout": 0.1,
  "enable_bias": true,
  "eos_token_id": 2,
  "ffn_dim": 4096,
  "hidden_size": 1024,
  "init_std": 0.02,
  "layer_norm_elementwise_affine": true,
  "layerdrop": 0.0,
  "max_position_embeddings": 2048,
  "model_type": "opt",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "prefix": "</s>",
  "torch_dtype": "float32",
  "transformers_version": "4.49.0",
  "use_cache": true,
  "vocab_size": 50272,
  "word_embed_proj_dim": 512
}

In [7]:
tokenizer

GPT2TokenizerFast(name_or_path='facebook/opt-350m', vocab_size=50265, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'bos_token': '</s>', 'eos_token': '</s>', 'unk_token': '</s>', 'pad_token': '<pad>'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	1: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=True, special=True),
}
)

In [8]:
tokenizer.eos_token

'</s>'

sacrebleu = evaluate.load("sacrebleu")
results_base = sacrebleu.compute(predictions=generated_outputs_base,
                                 references=expected_outputs)

print(list(results_base.keys()))
print(round(results_base["score"], 1))

In [9]:
!pip install torchinfo

In [10]:
from torchinfo import summary

In [11]:
summary(orig_model)

Layer (type:depth-idx)                             Param #
OPTForCausalLM                                     --
├─OPTModel: 1-1                                    --
│    └─OPTDecoder: 2-1                             --
│    │    └─Embedding: 3-1                         25,739,264
│    │    └─OPTLearnedPositionalEmbedding: 3-2     2,099,200
│    │    └─Linear: 3-3                            524,288
│    │    └─Linear: 3-4                            524,288
│    │    └─ModuleList: 3-5                        302,309,376
├─Linear: 1-2                                      25,739,264
Total params: 356,935,680
Trainable params: 356,935,680
Non-trainable params: 0

In [12]:
index = 200

dialogue = ds['test'][index]['dialogue']
summary = ds['test'][index]['summary']

prompt = f"""
Summarize the following conversation.

{dialogue}

Summary:
"""

inputs = tokenizer(prompt, return_tensors='pt')
output = tokenizer.decode(
    orig_model.generate(
        inputs["input_ids"], 
        max_new_tokens=200,
    )[0], 
    skip_special_tokens=True
)

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{prompt}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

---------------------------------------------------------------------------------------------------
INPUT PROMPT:

Summarize the following conversation.

#Person1#: Have you considered upgrading your system?
#Person2#: Yes, but I'm not sure what exactly I would need.
#Person1#: You could consider adding a painting program to your software. It would allow you to make up your own flyers and banners for advertising.
#Person2#: That would be a definite bonus.
#Person1#: You might also want to upgrade your hardware because it is pretty outdated now.
#Person2#: How can we do that?
#Person1#: You'd probably need a faster processor, to begin with. And you also need a more powerful hard disc, more memory and a faster modem. Do you have a CD-ROM drive?
#Person2#: No.
#Person1#: Then you might want to add a CD-ROM drive too, because most new software programs are coming out on Cds.
#Person2#: That sounds great. Thanks.

Summary:

-------------------------------------------------------------------

In [15]:
ds

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 12460
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 1500
    })
})

In [16]:
ds_1 = ds.filter(lambda example, index: index % 3 == 0, with_indices=True)
ds_1

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 4154
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 167
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
})

In [17]:
def sample_sort_ds(ds,tokenizer):
    df = ds.to_pandas()
    df['full_text'] = df.apply(lambda x: f'{x["dialogue"]}\n\n{x["summary"]}', axis=1)
    df['input_ids_len'] = df['full_text'].apply(lambda x: len(tokenizer(x, truncation=True).input_ids))
    df = df.sort_values(by=['input_ids_len'], ascending=True)
    df_1 = df[df['input_ids_len'] < 512].copy()
    df_1.reset_index(inplace=True, drop=True)
    df_1['id'] = range(len(df_1))
    df_2 = df_1[['id', 'dialogue', 'summary', 'topic']].copy()
    final_ds = hf_Dataset.from_pandas(df_2)
    return final_ds, df_1

In [18]:
ds_train, df_train = sample_sort_ds(ds_1['train'],tokenizer)
ds_train

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Dataset({
    features: ['id', 'dialogue', 'summary', 'topic'],
    num_rows: 4058
})

In [19]:
df_train.iloc[9]

id                                                               9
dialogue         #Person1#: This TV set is getting worse and wo...
summary          #Person2# suggests they buy a new TV set on sale.
topic                                             shopping on sale
full_text        #Person1#: This TV set is getting worse and wo...
input_ids_len                                                   76
Name: 9, dtype: object

In [20]:
ds_train[9]

{'id': 9,
 'dialogue': "#Person1#: This TV set is getting worse and worse. Now it doesn't work at all.\n#Person2#: Here's an advertisement on the newspaper about a big TV sale. Usually a big sale like this would have some good bargains. What would you say?",
 'summary': '#Person2# suggests they buy a new TV set on sale.',
 'topic': 'shopping on sale'}

In [21]:
ds_1['train'] = ds_train
ds_1['train']

Dataset({
    features: ['id', 'dialogue', 'summary', 'topic'],
    num_rows: 4058
})

In [22]:
ds_1['train'][4000], ds_train[4000]

({'id': 4000,
  'dialogue': "#Person1#: Hey, Jessica, there is a new fun test in the paper. I love to fill these things out.\n#Person2#: What's this one about?\n#Person1#: It's about health.\n#Person2#: OK. Read it to me. I'll keep score.\n#Person1#: OK. No. 1: Do you smoke more than ten cigarettes a day?\n#Person2#: That's easy. I gave up smoking three years ago.\n#Person1#: Right. You know, I should too.\n#Person2#: Yeah, I've heard that before.\n#Person1#: No, No, really. I'm going to. But for now I'd have to say, yes. OK. No. 2: Do you have a check-up at your doctor's office at least once a year?\n#Person2#: Yeah, the company makes us go to the doctor every year. How about you?\n#Person1#: Well, I went to the doctor...let's see...about three years ago.\n#Person2#: You should go more often.\n#Person1#: Well, let's move on to No. 7: Do you work more than ten hours a day?\n#Person2#: No, but you've been working a lot lately.\n#Person1#: I'm really tired. I should work a lot less. But 

In [23]:
ds_1['train'][12], ds_train[12]

({'id': 12,
  'dialogue': '#Person1#: Did you go see the doctor about your cough?\n#Person2#: The doctor said if I keep smoking it will increase my chance of having a heart attack or lung disease. And I am thinking about quitting smoking as the problems seem to be quite serious.',
  'summary': "#Person2#'s thinking about quitting smoking because of its harm to health.",
  'topic': "doctor's advice"},
 {'id': 12,
  'dialogue': '#Person1#: Did you go see the doctor about your cough?\n#Person2#: The doctor said if I keep smoking it will increase my chance of having a heart attack or lung disease. And I am thinking about quitting smoking as the problems seem to be quite serious.',
  'summary': "#Person2#'s thinking about quitting smoking because of its harm to health.",
  'topic': "doctor's advice"})

In [24]:
class dset(Dataset):
    def __init__(self, ds, start_prompt, end_prompt):
        super().__init__()
        self.data = ds
        self.start_prompt = start_prompt
        self.end_prompt = end_prompt

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        dialog = self.data[idx]['dialogue']
        summary = self.data[idx]['summary']
        x = self.start_prompt + dialog
        y = self.end_prompt + summary
        return x, y

In [25]:
start_prompt = 'Summarize the following conversation.\n\n'
end_prompt = '\n\nSummary: \n\n'

In [26]:
print(start_prompt)
print(end_prompt)

Summarize the following conversation.




Summary: 




In [27]:
ds_1

DatasetDict({
    train: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 4058
    })
    validation: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 167
    })
    test: Dataset({
        features: ['id', 'dialogue', 'summary', 'topic'],
        num_rows: 500
    })
})

In [28]:
train_ds = dset(ds_1['train'], start_prompt, end_prompt)
valid_ds = dset(ds_1['validation'], start_prompt, end_prompt)
test_ds = dset(ds_1['test'], start_prompt, end_prompt)

In [29]:
print(train_ds[0][0])
print("#############################")
print(train_ds[0][1])

Summarize the following conversation.

#Person1#: Did you hear that Anna needs to stay in bed for 4 weeks?
#Person2#: Yeah. She injured her spine in a fall and a doctor told her to lie flat on her back for a month so it can mend.
#############################


Summary: 

Anna needs to stay in bed for her spine injury.


In [30]:
print(train_ds[9][0])
print("#############################")
print(train_ds[9][1])

Summarize the following conversation.

#Person1#: This TV set is getting worse and worse. Now it doesn't work at all.
#Person2#: Here's an advertisement on the newspaper about a big TV sale. Usually a big sale like this would have some good bargains. What would you say?
#############################


Summary: 

#Person2# suggests they buy a new TV set on sale.


In [31]:
len(train_ds)

4058

In [32]:
print(valid_ds[0][0])
print("#############################")
print(valid_ds[0][1])

Summarize the following conversation.

#Person1#: Hello, how are you doing today?
#Person2#: I ' Ve been having trouble breathing lately.
#Person1#: Have you had any type of cold lately?
#Person2#: No, I haven ' t had a cold. I just have a heavy feeling in my chest when I try to breathe.
#Person1#: Do you have any allergies that you know of?
#Person2#: No, I don ' t have any allergies that I know of.
#Person1#: Does this happen all the time or mostly when you are active?
#Person2#: It happens a lot when I work out.
#Person1#: I am going to send you to a pulmonary specialist who can run tests on you for asthma.
#Person2#: Thank you for your help, doctor.
#############################


Summary: 

#Person2# has trouble breathing. The doctor asks #Person2# about it and will send #Person2# to a pulmonary specialist.


In [33]:
print(test_ds[3][0])
print("#############################")
print(test_ds[3][1])

Summarize the following conversation.

#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday
#############################


Summary: 

#Person1# and Brian are at the birthday party of Brian. Brian thinks #Person1# looks great and is popular.


In [34]:
eos_tok = tokenizer.eos_token
print(eos_tok)
eos_tok_id = [tokenizer.vocab[eos_tok]]
eos_tok_id

</s>


[2]

In [35]:
[tokenizer.vocab[tokenizer.eos_token]]

[2]

In [36]:
[tokenizer.vocab[tokenizer.bos_token]]

[2]

In [37]:
tokenizer.vocab[tokenizer.pad_token]

1

In [38]:
def collate_fn(batch, tokenizer):
    eos_tokenid = [tokenizer.vocab[tokenizer.eos_token]]
    pad_tokenid = [tokenizer.vocab[tokenizer.pad_token]]
    ignore_loss_id = [-100]
    ignore_token_id = [0]
    inp_list, att_list, y_list = [],  [], []

    for x, y in batch:
        x_tok = tokenizer(x, truncation=True).input_ids
        y_tok = tokenizer(y, truncation=True).input_ids

        prompt = x + y
        input_prompt = tokenizer(prompt, truncation=True)

        x_tok_len = len(x_tok)
        label_prompt = ignore_loss_id*(x_tok_len - 1) + y_tok[1:] + eos_tokenid
        #to remove extra start of sentence token from y_tok

        inp_list.append(input_prompt['input_ids'])
        att_list.append(input_prompt['attention_mask'])

        y_list.append(label_prompt)

    len_labs = [len(l) for l in y_list]
    max_len = max(len_labs)

    def pad_tokens_stack(in_list, max_len, padding):
        list_tensor = []
        for item in in_list:
            len_item = len(item)
            deficit = max_len - len_item
            if deficit > 0:
                item = item + padding*deficit
            list_tensor.append(torch.tensor(item))
        item_ids = torch.vstack(list_tensor)
        return item_ids

    input_ids = pad_tokens_stack(inp_list, max_len, pad_tokenid)
    attention_mask = pad_tokens_stack(att_list, max_len, ignore_token_id)
    labels = pad_tokens_stack(y_list, max_len, ignore_loss_id)

    return {'input_ids': input_ids, 'attention_mask': attention_mask}, labels

In [39]:
wrapper_collate_fn = partial(
        collate_fn,
        tokenizer=tokenizer
        )

In [40]:
cpu_cores = os.cpu_count()
train_dl = DataLoader(train_ds, shuffle=False, batch_size=3, 
                      num_workers=cpu_cores, collate_fn=wrapper_collate_fn)
valid_dl = DataLoader(valid_ds, shuffle=False, batch_size=3, 
                      num_workers=cpu_cores, collate_fn=wrapper_collate_fn)
test_dl = DataLoader(test_ds, shuffle=False, batch_size=3, 
                      num_workers=cpu_cores, collate_fn=wrapper_collate_fn)

In [41]:
def verify_dls(dl):
    iter_dl = iter(dl)
    i,l = next(iter_dl)
    print(i['input_ids'], i['input_ids'].shape)
    print(i['attention_mask'], i['attention_mask'].shape)
    print(l, l.shape)
    return i, l

In [42]:
i, l = verify_dls(train_dl)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


tensor([[    2, 38182,  3916,  2072,     5,   511,  1607,     4, 50118, 50118,
         10431, 41761,   134, 10431,    35,  6553,    47,  1798,    14,  7687,
           782,     7,  1095,    11,  3267,    13,   204,   688,   116, 50118,
         10431, 41761,   176, 10431,    35,  8976,     4,   264,  1710,    69,
         20625,    11,    10,  1136,     8,    10,  3299,   174,    69,     7,
          6105,  3269,    15,    69,   124,    13,    10,   353,    98,    24,
            64, 19300,     4, 50118, 50118, 47977,    35,  1437, 50118, 50118,
         35242,   782,     7,  1095,    11,  3267,    13,    69, 20625,  1356,
             4,     1,     1,     1,     1],
        [    2, 38182,  3916,  2072,     5,   511,  1607,     4, 50118, 50118,
         10431, 41761,   134, 10431,    35,   370,   356,  1341,   430,    31,
            99,    47,   341,     7,     4, 50118, 10431, 41761,   176, 10431,
            35,  9136,     4,    38,   554, 20203,  4595,    80,   107,   536,
       

In [43]:
verify_dls(valid_dl)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


tensor([[    2, 38182,  3916,  2072,     5,   511,  1607,     4, 50118, 50118,
         10431, 41761,   134, 10431,    35, 20920,     6,   141,    32,    47,
           608,   452,   116, 50118, 10431, 41761,   176, 10431,    35,    38,
           128, 10978,    57,   519,  3605,  9589, 12056,     4, 50118, 10431,
         41761,   134, 10431,    35,  6319,    47,    56,   143,  1907,     9,
          2569, 12056,   116, 50118, 10431, 41761,   176, 10431,    35,   440,
             6,    38,  2220,   128,   326,    56,    10,  2569,     4,    38,
            95,    33,    10,  2016,  2157,    11,   127,  7050,    77,    38,
           860,     7, 14575,     4, 50118, 10431, 41761,   134, 10431,    35,
          1832,    47,    33,   143, 26331,    14,    47,   216,     9,   116,
         50118, 10431, 41761,   176, 10431,    35,   440,     6,    38,   218,
           128,   326,    33,   143, 26331,    14,    38,   216,     9,     4,
         50118, 10431, 41761,   134, 10431,    35,  

({'input_ids': tensor([[    2, 38182,  3916,  2072,     5,   511,  1607,     4, 50118, 50118,
           10431, 41761,   134, 10431,    35, 20920,     6,   141,    32,    47,
             608,   452,   116, 50118, 10431, 41761,   176, 10431,    35,    38,
             128, 10978,    57,   519,  3605,  9589, 12056,     4, 50118, 10431,
           41761,   134, 10431,    35,  6319,    47,    56,   143,  1907,     9,
            2569, 12056,   116, 50118, 10431, 41761,   176, 10431,    35,   440,
               6,    38,  2220,   128,   326,    56,    10,  2569,     4,    38,
              95,    33,    10,  2016,  2157,    11,   127,  7050,    77,    38,
             860,     7, 14575,     4, 50118, 10431, 41761,   134, 10431,    35,
            1832,    47,    33,   143, 26331,    14,    47,   216,     9,   116,
           50118, 10431, 41761,   176, 10431,    35,   440,     6,    38,   218,
             128,   326,    33,   143, 26331,    14,    38,   216,     9,     4,
           5011

In [44]:
verify_dls(test_dl)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


tensor([[    2, 38182,  3916,  ...,  6257,  5988,     4],
        [    2, 38182,  3916,  ...,     1,     1,     1],
        [    2, 38182,  3916,  ...,     1,     1,     1]]) torch.Size([3, 381])
tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]) torch.Size([3, 381])
tensor([[-100, -100, -100,  ..., 5988,    4,    2],
        [-100, -100, -100,  ..., -100, -100, -100],
        [-100, -100, -100,  ..., -100, -100, -100]]) torch.Size([3, 381])


({'input_ids': tensor([[    2, 38182,  3916,  ...,  6257,  5988,     4],
          [    2, 38182,  3916,  ...,     1,     1,     1],
          [    2, 38182,  3916,  ...,     1,     1,     1]]),
  'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
          [1, 1, 1,  ..., 0, 0, 0],
          [1, 1, 1,  ..., 0, 0, 0]])},
 tensor([[-100, -100, -100,  ..., 5988,    4,    2],
         [-100, -100, -100,  ..., -100, -100, -100],
         [-100, -100, -100,  ..., -100, -100, -100]]))

In [46]:
out = orig_model(**i)
out

CausalLMOutputWithPast(loss=None, logits=tensor([[[  5.8232,   5.2200,  15.4210,  ...,   5.0798,   6.2076,   4.6757],
         [  1.2620,   0.7250,   7.5321,  ...,   0.4475,   1.9041,   2.6964],
         [  0.9085,   2.6399,   9.1968,  ...,   1.7405,   2.4824,  -0.7433],
         ...,
         [ -1.3695,  -2.1736,   6.9122,  ...,  -2.3306,  -1.7236,  -4.2469],
         [ -1.3695,  -2.1736,   6.9122,  ...,  -2.3306,  -1.7236,  -4.2469],
         [ -1.3695,  -2.1736,   6.9122,  ...,  -2.3306,  -1.7236,  -4.2469]],

        [[  5.8232,   5.2200,  15.4210,  ...,   5.0798,   6.2076,   4.6757],
         [  1.2620,   0.7250,   7.5321,  ...,   0.4475,   1.9041,   2.6964],
         [  0.9085,   2.6399,   9.1968,  ...,   1.7405,   2.4824,  -0.7433],
         ...,
         [  1.5367,  -0.0662,  13.7857,  ...,  -1.9630,  -1.5146,  -0.9540],
         [ -7.0488,  -7.8507,   6.0990,  ...,  -9.3783,  -6.7201,  -8.3996],
         [  0.9489,   0.2226,   8.1583,  ...,  -0.1467,   0.1554,  -1.8699]],

   

In [47]:
out.logits.shape

torch.Size([3, 85, 50272])

In [5]:
#Sagemaker related commands
sess = sagemaker.Session()
role = sagemaker.get_execution_role()
region = sess.boto_region_name
bucket = sess.default_bucket()
sess, role, region, bucket

[03/03/25 10:24:44] INFO     Found credentials from IAM Role:                                   ]8;id=384426;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=548755;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Found credentials from IAM Role:                                   ]8;id=990433;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=968178;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

(<sagemaker.session.Session at 0x7f7c2a110370>,
 'arn:aws:iam::191013407134:role/service-role/AmazonSageMaker-ExecutionRole-20250124T222384',
 'us-east-1',
 'sagemaker-us-east-1-191013407134')

In [6]:
%pwd

'/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/T5Summarization'

In [7]:
# let us get image uri
training_image_uri = sagemaker.image_uris.retrieve(
    framework='pytorch', 
    version='2.0',
    instance_type='ml.g4dn.xlarge',
    region=region,
    py_version='py310',
    image_scope='training'
)
print(training_image_uri)

763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.0-gpu-py310


In [8]:
objective_metric_name = "average training loss"
objective_type = "Minimize"
metric_definitions = [{"Name": "average training loss", "Regex": "Average loss: ([0-9\\.]+)"},
                      {"Name": "Perplexity", "Regex": "Perplexity: ([0-9\\.]+)"}]
objective_metric_name, objective_type, metric_definitions

('average training loss',
 'Minimize',
 [{'Name': 'average training loss', 'Regex': 'Average loss: ([0-9\\.]+)'},
  {'Name': 'Perplexity', 'Regex': 'Perplexity: ([0-9\\.]+)'}])

In [9]:
ic=1
i_type = "ml.g4dn.xlarge"
ic, i_type

(1, 'ml.g4dn.xlarge')

In [10]:
hparams = {
    'bs': 32,
    'lrate': 0.0008,
    'num_epochs': 2,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'facebook/opt-350m',
    'quant_bits': 4,
    'peft': "lora",
    'grad_accum_steps': 2 #set to 4 if cuda errors out
}
hparams

{'bs': 8,
 'lrate': 0.0008,
 'num_epochs': 2,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'facebook/opt-350m',
 'quant_bits': 4,
 'peft': 'lora',
 'grad_accum_steps': 4}

In [12]:
est_summ_4bit = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-fb-opt-Summariz.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_summ_4bit.fit(
    wait=True
    )

[03/02/25 13:48:18] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=581934;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=876068;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating training-job with name:                                       ]8;id=293528;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=255128;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             reutersgpt-train-2025-03-02-13-48-18-064                                              

2025-03-02 13:48:20 Starting - Starting the training job...
..25-03-02 13:48:34 Starting - Preparing the instances for training.
..25-03-02 13:49:02 Downloading - Downloading input data.
.................27 Downloading - Downloading the training image.
.bash: cannot set terminal process group (-1): Inappropriate ioctl for device..
bash: no job control in this shell
2025-03-02 13:53:05,041 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-03-02 13:53:05,061 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-03-02 13:53:05,071 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-03-02 13:53:05,078 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-03-02 13:53:06,988 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt:
/opt/conda/bin/python3.10 -m pip install -r requirements.txt
━━━━━━━━━━━━━━━━━━━━━━

In [18]:
hparams = {
    'bs': 8,
    'lrate': 0.0008,
    'num_epochs': 7,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'facebook/opt-350m',
    'quant_bits': 4,
    'peft': "lora",
    'grad_accum_steps': 8 #set to 4 if cuda errors out
}
hparams

{'bs': 8,
 'lrate': 0.0008,
 'num_epochs': 7,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'facebook/opt-350m',
 'quant_bits': 4,
 'peft': 'lora',
 'grad_accum_steps': 8}

In [19]:
est_summ_4bit = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-fb-opt-Summariz.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_summ_4bit.fit(
    wait=True
    )

[03/02/25 14:39:02] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=989799;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=826882;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[03/02/25 14:39:03] INFO     Creating training-job with name:                                       ]8;id=39819;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=91762;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             reutersgpt-train-2025-03-02-14-39-02-961                                              

2025-03-02 14:39:04 Starting - Starting the training job...
..25-03-02 14:39:19 Starting - Preparing the instances for training.
..25-03-02 14:39:46 Downloading - Downloading input data.
.................11 Downloading - Downloading the training image.
.bash: cannot set terminal process group (-1): Inappropriate ioctl for device..
bash: no job control in this shell
2025-03-02 14:43:42,247 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-03-02 14:43:42,266 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-03-02 14:43:42,276 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-03-02 14:43:42,284 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-03-02 14:43:44,487 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt:
/opt/conda/bin/python3.10 -m pip install -r requirements.txt
━━━━━━━━━━━━━━━━━━━━━━

In [20]:
!aws s3 ls --recursive {est_summ_4bit.model_data}
!aws s3 cp {est_summ_4bit.model_data} ./temp/
%cd temp
!ls -ltrh
!gunzip -dc model.tar.gz |tar xvf -
!ls -ltrh
%cd ..
%pwd

2025-03-02 15:33:13    7068877 reutersgpt-train-2025-03-02-14-39-02-961/output/model.tar.gz
download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2025-03-02-14-39-02-961/output/model.tar.gz to temp/model.tar.gz
/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/T5Summarization/temp
total 6.8M
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Mar  2 15:33 model.tar.gz
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_2.173195518073686/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_2.173195518073686/adapter_model.safetensors
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_2.173195518073686/tokenizer_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_2.173195518073686/training_args.bin
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_2.173195518073686/voc

'/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/T5Summarization'

In [21]:
hparams = {
    'bs': 8,
    'lrate': 0.0008,
    'num_epochs': 7,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'facebook/opt-350m',
    'quant_bits': 8,
    'peft': "lora",
    'grad_accum_steps': 8 #set to 4 if cuda errors out
}
hparams

{'bs': 8,
 'lrate': 0.0008,
 'num_epochs': 7,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'facebook/opt-350m',
 'quant_bits': 8,
 'peft': 'lora',
 'grad_accum_steps': 8}

In [22]:
est_summ_8bit = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-fb-opt-Summariz.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_summ_8bit.fit(
    wait=True
    )

[03/02/25 15:44:34] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=449123;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=22628;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating training-job with name:                                       ]8;id=763629;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=358709;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             reutersgpt-train-2025-03-02-15-44-34-037                                              

2025-03-02 15:44:35 Starting - Starting the training job...
..25-03-02 15:44:50 Starting - Preparing the instances for training.
..25-03-02 15:45:18 Downloading - Downloading input data.
.................43 Downloading - Downloading the training image.
.bash: cannot set terminal process group (-1): Inappropriate ioctl for device..
bash: no job control in this shell
2025-03-02 15:49:14,913 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-03-02 15:49:14,931 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-03-02 15:49:14,941 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-03-02 15:49:14,948 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-03-02 15:49:16,904 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt:
/opt/conda/bin/python3.10 -m pip install -r requirements.txt
━━━━━━━━━━━━━━━━━━━━━━

In [23]:
!aws s3 ls --recursive {est_summ_8bit.model_data}
!aws s3 cp {est_summ_8bit.model_data} ./temp/
%cd temp
!ls -ltrh
!gunzip -dc model.tar.gz |tar xvf -
!ls -ltrh

%cd ..
%pwd

2025-03-02 16:32:50    7071548 reutersgpt-train-2025-03-02-15-44-34-037/output/model.tar.gz
download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2025-03-02-15-44-34-037/output/model.tar.gz to temp/model.tar.gz
/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/T5Summarization/temp
total 6.8M
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2 15:32 ckpt_Epoch_7_Prplxty_2.173195518073686
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Mar  2 16:32 model.tar.gz
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_2.15419785550478/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_2.15419785550478/tokenizer_config.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_2.15419785550478/adapter_model.safetensors
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_2.15419785550478/training_args.bin
tar: Ignoring unknown extended

'/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/T5Summarization'

In [9]:
hparams = {
    'bs': 8,
    'lrate': 0.0008,
    'num_epochs': 7,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'facebook/opt-350m',
    'quant_bits': 0,
    'peft': "lora",
    'grad_accum_steps': 4 #set to 4 if cuda errors out
}
hparams

{'bs': 8,
 'lrate': 0.0008,
 'num_epochs': 7,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'facebook/opt-350m',
 'quant_bits': 0,
 'peft': 'lora',
 'grad_accum_steps': 4}

In [ ]:
est_summ_peft = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-fb-opt-Summariz.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_summ_peft.fit(
    wait=True
    )

[03/03/25 00:42:18] INFO     Found credentials from IAM Role:                                   ]8;id=838578;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=942370;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Found credentials from IAM Role:                                   ]8;id=321492;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=827820;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=748937;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=492345;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

[03/03/25 00:42:19] INFO     Creating training-job with name:                                       ]8;id=880111;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=96912;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             reutersgpt-train-2025-03-03-00-42-18-856                                              

2025-03-03 00:42:20 Starting - Starting the training job...
..25-03-03 00:42:35 Starting - Preparing the instances for training.
..25-03-03 00:43:06 Downloading - Downloading input data.
....................Downloading - Downloading the training image.
bash: cannot set terminal process group (-1): Inappropriate ioctl for devices..
bash: no job control in this shell
2025-03-03 00:47:30,017 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-03-03 00:47:30,035 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-03-03 00:47:30,047 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-03-03 00:47:30,055 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-03-03 00:47:31,876 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt:
/opt/conda/bin/python3.10 -m pip install -r requirements.txt
━━━━━━━━━━━━━━━━━━━━━━

In [11]:
est_summ_peft = Estimator.attach('reutersgpt-train-2025-03-03-00-42-18-856')


2025-03-03 01:35:34 Starting - Preparing the instances for training
2025-03-03 01:35:34 Downloading - Downloading the training image
2025-03-03 01:35:34 Training - Training image download completed. Training in progress.
2025-03-03 01:35:34 Uploading - Uploading generated training model
2025-03-03 01:35:34 Completed - Training job completed


In [13]:
!aws s3 ls --recursive {est_summ_peft.model_data}
!aws s3 cp {est_summ_peft.model_data} ./temp/
%cd temp
!ls -ltrh
!gunzip -dc model.tar.gz |tar xvf -
!ls -ltrh

%cd ..
%pwd


2025-03-03 01:35:30    7072193 reutersgpt-train-2025-03-03-00-42-18-856/output/model.tar.gz
download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2025-03-03-00-42-18-856/output/model.tar.gz to temp/model.tar.gz
/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/T5Summarization/temp
total 6.8M
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2 15:32 ckpt_Epoch_7_Prplxty_2.173195518073686
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2 16:32 ckpt_Epoch_7_Prplxty_2.15419785550478
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Mar  3 01:35 model.tar.gz
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_1.98545298512489/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_1.98545298512489/special_tokens_map.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7_Prplxty_1.98545298512489/vocab.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_7

'/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/T5Summarization'

In [10]:
hparams = {
    'bs': 8,
    'lrate': 0.0008,
    'num_epochs': 10,
    'environ': 'aws',
    'block_size': 256,
    'ckpt': 'facebook/opt-350m',
    'quant_bits': 8,
    'peft': "lora",
    'grad_accum_steps': 8 #set to 4 if cuda errors out
}
hparams

{'bs': 8,
 'lrate': 0.0008,
 'num_epochs': 10,
 'environ': 'aws',
 'block_size': 256,
 'ckpt': 'facebook/opt-350m',
 'quant_bits': 8,
 'peft': 'lora',
 'grad_accum_steps': 8}

In [11]:
est_summ_8bit_10e = Estimator(
    image_uri=training_image_uri,
    role=role,
    source_dir='./scripts',
    entry_point='train-fb-opt-Summariz.py',
    instance_count=ic,
    instance_type=i_type,
    base_job_name='reutersgpt-train',
    hyperparameters=hparams,
    metric_definitions=metric_definitions,
    objective_type=objective_type,
    objective_metric_name=objective_metric_name
)

est_summ_8bit_10e.fit(
    wait=True
    )

[03/03/25 10:25:12] INFO     Found credentials from IAM Role:                                   ]8;id=364097;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=864607;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

                    INFO     Found credentials from IAM Role:                                   ]8;id=134121;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py\credentials.py]8;;\:]8;id=986655;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/botocore/credentials.py#1132\1132]8;;\
                             BaseNotebookInstanceEc2InstanceRole                                                   

[03/03/25 10:25:13] INFO     SageMaker Python SDK will collect telemetry to help us better  ]8;id=146969;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=907446;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/telemetry/telemetry_logging.py#91\91]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#confi                        
                             guring-and-using-defaults-with-the-sagemaker-python-sdk.                              

                    INFO     Creating training-job with name:                                       ]8;id=677754;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py\session.py]8;;\:]8;id=615922;file:///home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/sagemaker/session.py#1042\1042]8;;\
                             reutersgpt-train-2025-03-03-10-25-13-549                                              

2025-03-03 10:25:15 Starting - Starting the training job...
..25-03-03 10:25:28 Starting - Preparing the instances for training.
....................Downloading - Downloading the training image.
bash: cannot set terminal process group (-1): Inappropriate ioctl for devices..
bash: no job control in this shell
2025-03-03 10:29:51,104 sagemaker-training-toolkit INFO     Imported framework sagemaker_pytorch_container.training
2025-03-03 10:29:51,123 sagemaker-training-toolkit INFO     No Neurons detected (normal if no neurons installed)
2025-03-03 10:29:51,133 sagemaker_pytorch_container.training INFO     Block until all host DNS lookups succeed.
2025-03-03 10:29:51,140 sagemaker_pytorch_container.training INFO     Invoking user training script.
2025-03-03 10:29:53,028 sagemaker-training-toolkit INFO     Installing dependencies from requirements.txt:
/opt/conda/bin/python3.10 -m pip install -r requirements.txt
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 6.1 MB/s eta 0:00:00
━━━━━

In [12]:
!aws s3 ls --recursive {est_summ_8bit_10e.model_data}
!aws s3 cp {est_summ_8bit_10e.model_data} ./temp/
%cd temp
!ls -ltrh
!gunzip -dc model.tar.gz |tar xvf -
!ls -ltrh

%cd ..
%pwd


2025-03-03 11:30:41    7071653 reutersgpt-train-2025-03-03-10-25-13-549/output/model.tar.gz
download: s3://sagemaker-us-east-1-191013407134/reutersgpt-train-2025-03-03-10-25-13-549/output/model.tar.gz to temp/model.tar.gz
/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/T5Summarization/temp
total 6.8M
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2 15:32 ckpt_Epoch_7_Prplxty_2.173195518073686
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  2 16:32 ckpt_Epoch_7_Prplxty_2.15419785550478
drwxr-xr-x 2 ec2-user ec2-user 4.0K Mar  3 01:35 ckpt_Epoch_7_Prplxty_1.98545298512489
-rw-rw-r-- 1 ec2-user ec2-user 6.8M Mar  3 11:30 model.tar.gz
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_10_Prplxty_1.8725549231207501/
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_10_Prplxty_1.8725549231207501/special_tokens_map.json
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
ckpt_Epoch_10_Prplxty_1.8725549231207501

'/home/ec2-user/SageMaker/transformersfromscratch/NLP_tasks/T5Summarization'